In [69]:

%pip install -q python-dotenv langchain langchain-community langchain-huggingface huggingface-hub 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()
import os
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [3]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated


c:\Anirudh\Langchain\campusx\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
exchangeRateAPI = os.getenv("EXCHANGE_RATE_API_KEY")
print(exchangeRateAPI)

e39ff65e9b6a9b39e08515cc


In [73]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/{exchangeRateAPI}/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [74]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1781913601,
 'time_last_update_utc': 'Sat, 20 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1782000001,
 'time_next_update_utc': 'Sun, 21 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.4312}

In [75]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [76]:
llm1  = HuggingFaceEndpoint(
        repo_id = "Qwen/Qwen2.5-72B-Instruct",
        task = "text-generation"
)

model1 = ChatHuggingFace(llm=llm1)

model_with_tools = model1.bind_tools([get_conversion_factor, convert])

In [77]:
messages = [HumanMessage('What is the conversion factor between USD to INR, and based on that can you convert 10 usd to inr')]

In [78]:
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)
messages

[HumanMessage(content='What is the conversion factor between USD to INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_ZLkIdHULHElHjkms4Njdzj8k', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_tELfugEjymunlcXNAafT55I7', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 446, 'total_tokens': 498}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ee4a1-0df7-7fb1-9c4a-9bfa897712f4-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_ZLkIdHULHElHjkms4Njdzj8k

In [79]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool call and get the value of the conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        # fetch this converson rate
        conversion_rate = (json.loads(tool_message1.content)['conversion_rate'])
        #append this tool message to messages list
        messages.append(tool_message1)
    # execute the second tool using the conversion rate from tool1
    if tool_call['name'] == 'convert':
        # fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [81]:
final_answer = model1.invoke(messages)

In [83]:
print(final_answer.content)

The current conversion rate from USD to INR is 94.4312. Therefore, 10 USD is equivalent to 944.312 INR.
